# Gold template: Type 1 dimension (Snowflake)

Builds a dimension that keeps only the current value of each attribute: a change overwrites it.

**Standards this notebook follows**

1. **Surrogate key from Silver.** The dimension's key is the Silver row's `<NAME>_SK` from the key table, so a
   rebuild never renumbers it and facts never need re-keying.
2. **Unknown member.** Row `-1` always exists, with `Unknown` in its text columns.
3. **Business key kept** as an attribute. **Only descriptive attributes**, no measures.
4. **Upsert, never overwrite.** `MERGE` on the key, updating a row only when its `_ROW_HASH` changed. Rows
   deleted in Silver stay, because facts may still point at them.
5. **Audit columns:** `_ROW_HASH`, `_RUN_ID`, `_LOADED_AT`. **Layers.** one database per layer: `BRONZE`, `SILVER`, `GOLD`, with keys in `KEYS`. Tables are named `<DATABASE>.<SCHEMA>.<TABLE>`.

## Who supports this notebook

This is an **enterprise template** (`gold-dim-type1` 1.0.0, Snowflake). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=gold-dim-type1 platform=snowflake version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "SILVER.CRM.CUSTOMERS"
target_table = "GOLD.SALES.DIM_CUSTOMER"
sk_column = "customer_sk"            # the Silver key from the key table; it is this dimension's key
business_key = ["customer_id"]
attributes = {                       # every column of the dimension besides the key, and its type
    "customer_id": "number(10,0)",
    "name": "varchar",
    "segment": "varchar",
    "city": "varchar",
    "email": "varchar",
}
run_id = ""          # the task or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=snowflake version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
import re
import uuid
from datetime import datetime, timezone

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSQLException

session = get_active_session()
UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_NAME = re.compile(r"[A-Za-z_][A-Za-z0-9_$]{0,254}")
_TYPE = re.compile(r"(?i)(varchar|string|text|boolean|number\(\d{1,2}(,\s*\d{1,2})?\)|int|integer|bigint|float|double|date|timestamp_ntz|timestamp_tz)")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name, checked: parameters arrive from tasks, so nothing else can reach the SQL."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 3 or not all(_NAME.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(parts).upper()


def c(name: str) -> str:
    if not _NAME.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return name.upper()


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip().upper()


def run(sql: str) -> list:
    """Runs one statement. DML and DDL return one summary row; collect() is how Snowpark runs them."""
    return session.sql(sql).collect()


def table_exists(name: str) -> bool:
    try:
        session.table(q(name)).schema  # resolved now, so a missing table raises here
        return True
    except SnowparkSQLException:
        return False


def row_hash(columns: list, alias: str = "") -> str:
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    a = f"{alias}." if alias else ""
    parts = [f"COALESCE(TO_VARCHAR({a}{c(x)}), CHR(0))" for x in columns] or ["''"]
    return f"SHA2(ARRAY_TO_STRING(ARRAY_CONSTRUCT({', '.join(parts)}), CHR(31)), 256)"


def is_text(dtype: str) -> bool:
    return dtype.upper().split("(")[0] in ("VARCHAR", "STRING", "TEXT")


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
LOADED_AT = f"TO_TIMESTAMP_NTZ('{loaded_at}')"
print(f"Run {run_id} at {loaded_at} UTC")

## 1. Create the table and its Unknown member

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=snowflake version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
for name in business_key:
    if name not in attributes:
        raise ValueError(f"The business key column {name} must be one of the attributes")
attributes = {c(n): checked_type(t) for n, t in attributes.items()}
business_key = [c(k) for k in business_key]
sk = c(sk_column)
every = [sk, *attributes, "_ROW_HASH", "_RUN_ID", "_LOADED_AT"]
def unknown_member_values(types: dict) -> list:
    """The Unknown member's value for each column: 'Unknown' for text, NULL for the rest."""
    return ["'Unknown'" if is_text(t) else f"CAST(NULL AS {t})" for t in types.values()]

run(f"""CREATE TABLE IF NOT EXISTS {q(target_table)} (
    {", ".join([f"{sk} NUMBER(10,0) NOT NULL", *[f"{n} {t}" for n, t in attributes.items()],
                "_ROW_HASH VARCHAR(64)", "_RUN_ID VARCHAR", "_LOADED_AT TIMESTAMP_NTZ"])},
    CONSTRAINT PK_{sk} PRIMARY KEY ({sk}))""")

run(f"""MERGE INTO {q(target_table)} AS t
USING (SELECT {UNKNOWN_SK} AS {sk}, {", ".join(f"{x} AS {n}" for x, n in zip(unknown_member_values(attributes), attributes))},
              'unknown-member' AS _ROW_HASH, '{run_id}' AS _RUN_ID, {LOADED_AT} AS _LOADED_AT) AS s
ON t.{sk} = s.{sk}
WHEN NOT MATCHED THEN INSERT ({", ".join(every)}) VALUES ({", ".join(f"s.{n}" for n in every)})""")

## 2. Upsert from Silver

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=snowflake version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
result = run(f"""MERGE INTO {q(target_table)} AS t
USING (
    SELECT {sk}, {", ".join(f"CAST({n} AS {t}) AS {n}" for n, t in attributes.items())},
           {row_hash(list(attributes))} AS _ROW_HASH, '{run_id}' AS _RUN_ID, {LOADED_AT} AS _LOADED_AT
    FROM {q(source_table)} WHERE {sk} <> {UNKNOWN_SK}
) AS s
ON t.{sk} = s.{sk}
WHEN MATCHED AND t._ROW_HASH <> s._ROW_HASH THEN
    UPDATE SET {", ".join(f"t.{n} = s.{n}" for n in every if n != sk)}
WHEN NOT MATCHED THEN
    INSERT ({", ".join(every)}) VALUES ({", ".join(f"s.{n}" for n in every)})""")
print(f"Dimension MERGE: {result[0].as_dict() if result else 'done'}")

## 3. Check the result

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=snowflake version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
bk = ", ".join(business_key)
summary = run(f"""SELECT COUNT(*) AS ROW_COUNT, COUNT(DISTINCT {sk}) AS KEY_COUNT,
       COUNT_IF({sk} = {UNKNOWN_SK}) AS UNKNOWN_MEMBERS FROM {q(target_table)}""")[0]
duplicates = run(f"""SELECT {bk} FROM {q(target_table)} WHERE {sk} <> {UNKNOWN_SK}
GROUP BY {bk} HAVING COUNT(*) > 1 LIMIT 5""")
if summary["ROW_COUNT"] != summary["KEY_COUNT"] or summary["UNKNOWN_MEMBERS"] != 1 or duplicates:
    raise RuntimeError(f"{target_table} breaks a dimension rule: {summary.as_dict()}, duplicate business keys {duplicates}")
print(f"{target_table}: {summary['ROW_COUNT']} rows including the Unknown member")